Cell 1 — Mount Google Drive

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# Experiment: MMS-Tarifit zero-shot evaluation
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Cell 2 — Define project paths

In [ ]:
# ============================================================
# CELL 2: Define project and validation paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

VALIDATION_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1_1"
    / "validation.csv"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "mms_tarifit"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project:", PROJECT_ROOT.exists())
print("Validation CSV:", VALIDATION_CSV.exists())

Project: True
Validation CSV: True


Cell 3 — Install dependencies

In [ ]:
# ============================================================
# CELL 3: Install MMS evaluation dependencies
# ============================================================

!pip install -q transformers datasets jiwer soundfile accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 76.7 MB/s eta 0:00:00


Cell 4 — Load validation metadata

In [ ]:
# ============================================================
# CELL 4: Load Corpus V1.1 validation metadata
# ============================================================

import pandas as pd

validation_df = pd.read_csv(
    VALIDATION_CSV
)

print("Validation segments:", len(validation_df))

print(
    "Duration:",
    round(
        validation_df["duration_seconds"].sum() / 60,
        2
    ),
    "minutes"
)

print(
    "Speakers:",
    validation_df["speaker_group_id"].unique()
)

validation_df.head()

Validation segments: 133
Duration: 18.12 minutes
Speakers: ['SPK007' 'SPK010']


,segment_id,recording_id,speaker_group_id,audio_path,duration_seconds,transcription,absolute_audio_path
0,REC090_SEG0010,REC090,SPK007,data/processed/segments/REC090/REC090_SEG0010.wav,3.280,ssalamuɛlikum necc meryem,/Users/mac/masterAI/tarifit_asr_tfm/data/proce...
1,REC090_SEG0011,REC090,SPK007,data/processed/segments/REC090/REC090_SEG0011.wav,5.200,aqay ruxxa tnayn uɛecrin sana di hulanda,/Users/mac/masterAI/tarifit_asr_tfm/data/proce...
2,REC090_SEG0012,REC090,SPK007,data/processed/segments/REC090/REC090_SEG0012.wav,12.048,mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ...,/Users/mac/masterAI/tarifit_asr_tfm/data/proce...
3,REC090_SEG0013,REC090,SPK007,data/processed/segments/REC090/REC090_SEG0013.wav,8.080,umi wsiɣd dda ufix manayenni wa ǧi ca min ira ...,/Users/mac/masterAI/tarifit_asr_tfm/data/proce...
4,REC090_SEG0014,REC090,SPK007,data/processed/segments/REC090/REC090_SEG0014.wav,7.160,a nec mammec ira ǧjix di lmeɣrib waǧi manayenn...,/Users/mac/masterAI/tarifit_asr_tfm/data/proce...


Cell 5 — Check CPU/GPU

In [ ]:
# ============================================================
# CELL 5: Check available device
# ============================================================

import torch

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

if DEVICE == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Device: cuda
GPU: Tesla T4


Cell 6 — Load MMS-Tarifit model

In [ ]:
# ============================================================
# CELL 6: Load MMS-Tarifit ASR model
# ============================================================

from transformers import (
    AutoProcessor,
    AutoModelForCTC
)

MMS_MODEL_NAME = (
    "iukocha/mms-tachebdant-from-tarifit"
)

processor = AutoProcessor.from_pretrained(
    MMS_MODEL_NAME
)

model = AutoModelForCTC.from_pretrained(
    MMS_MODEL_NAME
)

model.to(DEVICE)
model.eval()

print("Model loaded:")
print(MMS_MODEL_NAME)

print(
    "Parameters:",
    round(
        model.num_parameters() / 1e6,
        1
    ),
    "M"
)

print(
    "Vocabulary size:",
    model.config.vocab_size
)

processor_config.json:   0%|          | 0.00/299 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

Model loaded:
iukocha/mms-tachebdant-from-tarifit
Parameters: 964.7 M
Vocabulary size: 46


Cell 7 — Inspect the MMS tokenizer vocabulary

In [ ]:
# ============================================================
# CELL 7: Inspect MMS-Tarifit tokenizer vocabulary
# ============================================================

vocab = processor.tokenizer.get_vocab()

print("Vocabulary size:", len(vocab))

print("\nVocabulary tokens:")
print(
    sorted(
        vocab.keys(),
        key=lambda x: vocab[x]
    )
)

Vocabulary size: 46

Vocabulary tokens:
['!', ',', '-', '.', '?', 'a', 'b', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'u', 'w', 'x', 'y', 'z', 'ç', 'ā', 'č', 'ŋ', 'š', 'ə', 'ɛ', 'ɣ', 'ḍ', 'ḥ', 'ṣ', 'ṭ', '[UNK]', 'ẓ', '[PAD]', '<s>', '</s>', ' ']


Cell 8 — Load one validation audio sample

In [ ]:
# ============================================================
# CELL 8: Load one validation audio sample
# ============================================================

import soundfile as sf

row = validation_df.iloc[6]

audio_file = (
    PROJECT_ROOT
    / row["audio_path"]
)

audio, sampling_rate = sf.read(
    audio_file
)

print("Segment:")
print(row["segment_id"])

print("\nAudio:")
print(audio_file)

print("\nSample rate:")
print(sampling_rate)

print("\nReference:")
print(row["transcription"])

Segment:
REC090_SEG0016

Audio:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC090/REC090_SEG0016.wav

Sample rate:
16000

Reference:
di lmeɣrib neccin mammec ira niɛicc


Cell 9 — Run one MMS-Tarifit prediction

In [ ]:
# ============================================================
# CELL 9: Run one MMS-Tarifit zero-shot prediction
# ============================================================

import torch

inputs = processor(
    audio,
    sampling_rate=sampling_rate,
    return_tensors="pt"
)

input_values = (
    inputs["input_values"]
    .to(DEVICE)
)

attention_mask = (
    inputs.get("attention_mask")
)

if attention_mask is not None:
    attention_mask = (
        attention_mask.to(DEVICE)
    )

with torch.no_grad():

    outputs = model(
        input_values=input_values,
        attention_mask=attention_mask
    )

predicted_ids = torch.argmax(
    outputs.logits,
    dim=-1
)

prediction = processor.batch_decode(
    predicted_ids
)[0]

print("REFERENCE:")
print(row["transcription"])

print("\nMMS-TARIFIT:")
print(prediction)

REFERENCE:
di lmeɣrib neccin mammec ira niɛicc

MMS-TARIFIT:
di lməɣrim, nəššin mamšira niɛišə.


Cell 10 — Run MMS on 10 validation samples

In [ ]:
# ============================================================
# CELL 10: Run MMS-Tarifit on 10 validation samples
# ============================================================

import torch
import soundfile as sf

sample_indices = list(range(10))

mms_results = []

for idx in sample_indices:

    row = validation_df.iloc[idx]

    audio_file = PROJECT_ROOT / row["audio_path"]

    audio, sampling_rate = sf.read(audio_file)

    inputs = processor(
        audio,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )

    input_values = inputs["input_values"].to(DEVICE)

    attention_mask = inputs.get("attention_mask")

    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    with torch.no_grad():
        outputs = model(
            input_values=input_values,
            attention_mask=attention_mask
        )

    predicted_ids = torch.argmax(
        outputs.logits,
        dim=-1
    )

    prediction = processor.batch_decode(
        predicted_ids
    )[0]

    mms_results.append({
        "segment_id": row["segment_id"],
        "speaker": row["speaker_group_id"],
        "reference": row["transcription"],
        "prediction_raw": prediction
    })

print("Finished:", len(mms_results), "samples")

Finished: 10 samples


Cell 11 — Inspect the 10 predictions

In [ ]:
# ============================================================
# CELL 11: Inspect 10 MMS-Tarifit predictions
# ============================================================

for i, item in enumerate(mms_results):

    print("=" * 80)
    print(f"EXAMPLE {i + 1}")
    print("Segment:", item["segment_id"])
    print("Speaker:", item["speaker"])

    print("\nREFERENCE:")
    print(item["reference"])

    print("\nMMS RAW:")
    print(item["prediction_raw"])

    print()

EXAMPLE 1
Segment: REC090_SEG0010
Speaker: SPK007

REFERENCE:
ssalamuɛlikum necc meryem

MMS RAW:
ssalamuɛlikum, nəš məryəm.

EXAMPLE 2
Segment: REC090_SEG0011
Speaker: SPK007

REFERENCE:
aqay ruxxa tnayn uɛecrin sana di hulanda

MMS RAW:
aqay ruxa tnən uɛəšrin sana d i hulanda.

EXAMPLE 3
Segment: REC090_SEG0012
Speaker: SPK007

REFERENCE:
mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca

MMS RAW:
məršəx ak misənjjiran, usiɣ d zi lməɣrib, umiraqqa ydi lməɣribirəqqa ɣas ad rəḥɣar urupa ad ggəx adəgəxmaša.

EXAMPLE 4
Segment: REC090_SEG0013
Speaker: SPK007

REFERENCE:
umi wsiɣd dda ufix manayenni wa ǧi ca min ira ɣari ddhi rɛqel inu

MMS RAW:
umi wsiɣ dda, ufi x manayənni wa ji ša miriraɣari ddhi lɛqəl inu.

EXAMPLE 5
Segment: REC090_SEG0014
Speaker: SPK007

REFERENCE:
a nec mammec ira ǧjix di lmeɣrib waǧi manayenni uffix dda

MMS RAW:
a nəšmamši ra ji x di lməɣrib, waji manayənni ufi x dda.

EXAMPLE 6
Segment: REC090_SE

Cell 12 — Define a first MMS orthographic normalization

In [ ]:
# ============================================================
# CELL 12: Define MMS-to-Corpus-V1.1 normalization
# ============================================================

import re
import unicodedata

def normalize_mms_output(text):

    text = unicodedata.normalize("NFC", str(text))
    text = text.lower()

    # Known MMS orthographic differences
    text = text.replace("ə", "e")
    text = text.replace("š", "c")

    # Remove punctuation
    text = re.sub(
        r"[!?,.\-]",
        " ",
        text
    )

    # Collapse repeated spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


for item in mms_results:
    item["prediction_normalized"] = normalize_mms_output(
        item["prediction_raw"]
    )

print("Normalization ready.")

Normalization ready.


Cell 13 — Compare raw and normalized MMS output

In [ ]:
# ============================================================
# CELL 13: Compare raw vs normalized MMS predictions
# ============================================================

for i, item in enumerate(mms_results):

    print("=" * 80)
    print(f"EXAMPLE {i + 1}")

    print("\nREFERENCE:")
    print(item["reference"])

    print("\nMMS RAW:")
    print(item["prediction_raw"])

    print("\nMMS NORMALIZED:")
    print(item["prediction_normalized"])

    print()

EXAMPLE 1

REFERENCE:
ssalamuɛlikum necc meryem

MMS RAW:
ssalamuɛlikum, nəš məryəm.

MMS NORMALIZED:
ssalamuɛlikum nec meryem

EXAMPLE 2

REFERENCE:
aqay ruxxa tnayn uɛecrin sana di hulanda

MMS RAW:
aqay ruxa tnən uɛəšrin sana d i hulanda.

MMS NORMALIZED:
aqay ruxa tnen uɛecrin sana d i hulanda

EXAMPLE 3

REFERENCE:
mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca

MMS RAW:
məršəx ak misənjjiran, usiɣ d zi lməɣrib, umiraqqa ydi lməɣribirəqqa ɣas ad rəḥɣar urupa ad ggəx adəgəxmaša.

MMS NORMALIZED:
mercex ak misenjjiran usiɣ d zi lmeɣrib umiraqqa ydi lmeɣribireqqa ɣas ad reḥɣar urupa ad ggex adegexmaca

EXAMPLE 4

REFERENCE:
umi wsiɣd dda ufix manayenni wa ǧi ca min ira ɣari ddhi rɛqel inu

MMS RAW:
umi wsiɣ dda, ufi x manayənni wa ji ša miriraɣari ddhi lɛqəl inu.

MMS NORMALIZED:
umi wsiɣ dda ufi x manayenni wa ji ca miriraɣari ddhi lɛqel inu

EXAMPLE 5

REFERENCE:
a nec mammec ira ǧjix di lmeɣrib waǧi manayenni uff

Cell 14 — Preliminary raw vs normalized WER/CER

In [ ]:
# ============================================================
# CELL 14: Preliminary MMS WER/CER on 10 samples
# ============================================================

from jiwer import wer, cer

references = [
    item["reference"]
    for item in mms_results
]

raw_predictions = [
    item["prediction_raw"]
    for item in mms_results
]

normalized_predictions = [
    item["prediction_normalized"]
    for item in mms_results
]

print("RAW MMS")
print(
    "WER:",
    round(wer(references, raw_predictions) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(references, raw_predictions) * 100, 2),
    "%"
)

print("\nNORMALIZED MMS")
print(
    "WER:",
    round(wer(references, normalized_predictions) * 100, 2),
    "%"
)
print(
    "CER:",
    round(cer(references, normalized_predictions) * 100, 2),
    "%"
)

RAW MMS
WER: 84.91 %
CER: 26.38 %

NORMALIZED MMS
WER: 65.41 %
CER: 15.83 %


Cell 15 — Run MMS on the full validation set

In [ ]:
# ============================================================
# CELL 15: Run MMS-Tarifit on full validation set
# ============================================================

import torch
import soundfile as sf
from tqdm.auto import tqdm

mms_full_results = []

for _, row in tqdm(
    validation_df.iterrows(),
    total=len(validation_df)
):

    audio_file = PROJECT_ROOT / row["audio_path"]

    audio, sampling_rate = sf.read(audio_file)

    inputs = processor(
        audio,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )

    input_values = inputs["input_values"].to(DEVICE)

    attention_mask = inputs.get("attention_mask")

    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)

    with torch.no_grad():
        outputs = model(
            input_values=input_values,
            attention_mask=attention_mask
        )

    predicted_ids = torch.argmax(
        outputs.logits,
        dim=-1
    )

    prediction_raw = processor.batch_decode(
        predicted_ids
    )[0]

    prediction_normalized = normalize_mms_output(
        prediction_raw
    )

    mms_full_results.append({
        "segment_id": row["segment_id"],
        "speaker_group_id": row["speaker_group_id"],
        "reference": row["transcription"],
        "prediction_raw": prediction_raw,
        "prediction_normalized": prediction_normalized,
    })

print("Finished:", len(mms_full_results))

  0%|          | 0/133 [00:00<?, ?it/s]

Finished: 133


Cell 16 — Compute full-validation WER/CER

In [ ]:
# ============================================================
# CELL 16: Full validation raw vs normalized MMS metrics
# ============================================================

from jiwer import wer, cer

references = [
    x["reference"]
    for x in mms_full_results
]

raw_predictions = [
    x["prediction_raw"]
    for x in mms_full_results
]

normalized_predictions = [
    x["prediction_normalized"]
    for x in mms_full_results
]

raw_wer = wer(references, raw_predictions) * 100
raw_cer = cer(references, raw_predictions) * 100

norm_wer = wer(references, normalized_predictions) * 100
norm_cer = cer(references, normalized_predictions) * 100

print("FULL VALIDATION — RAW MMS")
print(f"WER: {raw_wer:.2f}%")
print(f"CER: {raw_cer:.2f}%")

print("\nFULL VALIDATION — NORMALIZED MMS")
print(f"WER: {norm_wer:.2f}%")
print(f"CER: {norm_cer:.2f}%")

FULL VALIDATION — RAW MMS
WER: 93.78%
CER: 53.22%

FULL VALIDATION — NORMALIZED MMS
WER: 81.03%
CER: 45.69%


Cell 17 — Save predictions to CSV

In [ ]:
# ============================================================
# CELL 17: Save full MMS validation predictions
# ============================================================

import pandas as pd

mms_results_df = pd.DataFrame(
    mms_full_results
)

OUTPUT_CSV = (
    RESULTS_DIR
    / "mms_tarifit_validation_predictions.csv"
)

mms_results_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8"
)

print("Saved to:")
print(OUTPUT_CSV)

Saved to:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/mms_tarifit/mms_tarifit_validation_predictions.csv
